# WaveForge GPU vs Meep CPU — Brain Clot MIMO Benchmark

**Compares:**
- Meep CPU: pre-measured locally (brain clot, 16-antenna MIMO, 1 GHz)
- WaveForge GPU: measured here on Colab T4

Same scenario: 150×150 grid, 16 TX antennas, 800 steps/TX, 1 GHz Ricker

> Runtime → Change runtime type → **T4 GPU**

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True,text=True)
print('GPU:', r.stdout.strip() if r.returncode==0 else 'NOT DETECTED')


In [ ]:
!git clone https://github.com/shahzaibshazoo/waveforge.git
!pip install torch numpy matplotlib --quiet


In [ ]:
import sys, time, math
import numpy as np
sys.path.insert(0, '/content/waveforge/src')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'
print(f'PyTorch {torch.__version__} | Device: {DEVICE} | GPU: {GPU_NAME}')


## Part 1: WaveForge GPU — Brain Clot MIMO (16 TX)

In [ ]:
from core import YeeGrid, FieldSet, MurABC, FDTD2D
from core import RickerWavelet, PointSource, SourceCollection
from core.materials import MaterialMap, Material, TISSUE_LIBRARY

NX, NY = 150, 150
DX     = 2e-3
N_STEPS = 800
N_TX    = 16
FREQ    = 1e9
ARRAY_R = 65
CENTER  = (75, 75)

CLOT_MAT = Material('clot', eps_r=70.0, sigma=3.0)

def build_brain_grid():
    grid = YeeGrid(NX, NY, dx=DX, dy=DX, device=DEVICE)
    mm = MaterialMap(grid, default=TISSUE_LIBRARY['free_space'])
    mm.add_circle(CENTER, 55, TISSUE_LIBRARY['skull'])
    mm.add_circle(CENTER, 51, TISSUE_LIBRARY['brain'])
    mm.add_circle((75, 93), 14, CLOT_MAT)  # front clot
    Ca, Cb = mm.build()
    return grid, Ca, Cb

def get_antennas(n, r, cx, cy):
    return [(int(round(cx + r*math.cos(2*math.pi*k/n))),
             int(round(cy + r*math.sin(2*math.pi*k/n)))) for k in range(n)]

print('Building brain phantom...')
grid, Ca, Cb = build_brain_grid()
ants = get_antennas(N_TX, ARRAY_R, *CENTER)
print(f'Grid: {NX}x{NY}, device={DEVICE}, antennas={N_TX}')


In [ ]:
# Time ONE TX run (representative of all 16)
def time_one_tx(grid, Ca, Cb, tx_pos, n_steps, warmup=20):
    fields   = FieldSet(grid)
    boundary = MurABC(grid, fields.Hz)
    wv  = RickerWavelet(amplitude=1.0, peak_freq=FREQ, t0=1.5/FREQ)
    src = PointSource(wv, tx_pos[0], tx_pos[1], 'Hz', grid=grid, N_steps=warmup+n_steps)
    sim = FDTD2D(grid, fields, boundary, SourceCollection([src]), Ca=Ca, Cb=Cb, n_check=1000)
    sim.run(warmup)  # warmup
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    sim.run(n_steps)
    if DEVICE=='cuda': torch.cuda.synchronize()
    return time.perf_counter() - t0

print('Timing one TX run...')
t_one = time_one_tx(grid, Ca, Cb, ants[0], N_STEPS)
mcells_s = N_STEPS * NX * NY / t_one / 1e6
t_full_mimo = t_one * N_TX

print(f'One TX run: {t_one:.2f}s')
print(f'Throughput: {mcells_s:.1f} Mcells/s')
print(f'Full MIMO (16 TX) estimate: {t_full_mimo:.1f}s ({t_full_mimo/60:.1f} min)')

gpu_result = {'mcells_s': round(mcells_s,2), 't_one_tx_s': round(t_one,3),
              't_full_mimo_s': round(t_full_mimo,1), 'gpu': GPU_NAME}


## Part 2: Load Pre-Measured Meep CPU Results

In [ ]:
import json

# Meep CPU brain clot benchmark (measured locally with pymeep)
meep_brain_result = {
    'mcells_s':     16.09,   # Mcells/s on laptop CPU
    't_one_tx_s':   1.12,    # seconds for one TX run
    't_full_mimo_s': 17.9,   # 16 TX * 1.12s
    'platform':     'Laptop CPU (pymeep)',
    'grid':         '150x150',
    'n_steps':       800,
    'n_tx':          16,
}

speedup = gpu_result['mcells_s'] / meep_brain_result['mcells_s']
time_speedup = meep_brain_result['t_full_mimo_s'] / gpu_result['t_full_mimo_s']

print('='*55)
print('  Brain Clot MIMO Benchmark — WaveForge vs Meep')
print('='*55)
print(f'  Scenario:   150x150, 16-antenna MIMO, 1GHz, 800 steps')
print(f'  WaveForge GPU ({GPU_NAME}):  {gpu_result["mcells_s"]:>8.1f} Mcells/s  {gpu_result["t_full_mimo_s"]:>6.1f}s MIMO')
print(f'  Meep CPU (laptop):          {meep_brain_result["mcells_s"]:>8.1f} Mcells/s  {meep_brain_result["t_full_mimo_s"]:>6.1f}s MIMO')
print(f'  Speedup (throughput):    {speedup:>6.1f}x')
print(f'  Speedup (total MIMO time): {time_speedup:>5.1f}x')
print('='*55)


## Part 3: Comparison Chart

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Brain Clot MIMO Imaging — WaveForge ({GPU_NAME}) vs Meep CPU',
             fontsize=12, fontweight='bold')

# Left: Throughput
ax = axes[0]
vals  = [gpu_result['mcells_s'], meep_brain_result['mcells_s']]
names = [f'WaveForge\n({GPU_NAME})', 'Meep\n(CPU)']
cols  = ['#2ca02c', '#d62728']
bars = ax.bar(names, vals, color=cols, alpha=0.85, width=0.45)
ax.bar_label(bars, [f'{v:.1f}\nMcells/s' for v in vals], fontsize=11, padding=4)
ax.set(ylabel='Throughput (Mcells/s)', title='Throughput Comparison')
ax.set_ylim(0, max(vals)*1.4)
ax.grid(True, alpha=0.3, axis='y')

# Right: Total MIMO time
ax2 = axes[1]
times = [gpu_result['t_full_mimo_s'], meep_brain_result['t_full_mimo_s']]
bars2 = ax2.bar(names, times, color=cols, alpha=0.85, width=0.45)
ax2.bar_label(bars2, [f'{v:.1f}s' for v in times], fontsize=11, padding=4)
ax2.set(ylabel='Time for full MIMO (16 TX) (s)', title=f'Total Simulation Time\n{speedup:.1f}x throughput, {time_speedup:.1f}x faster end-to-end')
ax2.set_ylim(0, max(times)*1.35)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
chart = '/content/brain_clot_comparison.png'
fig.savefig(chart, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {chart}')


## Part 4: Commit Results to GitHub

In [ ]:
# Paste your GitHub Personal Access Token here
# Get one at: GitHub → Settings → Developer settings → Personal access tokens → Fine-grained
# Select repo: waveforge, permission: Contents → Read and write
# Use Colab Secrets (left sidebar 🔑) — add GITHUB_TOKEN
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print('Token loaded from Colab Secrets')
except:
    GITHUB_TOKEN = ''  # paste here if not using Secrets
    print('Paste your GitHub token into GITHUB_TOKEN above')

import os, shutil, datetime
os.chdir('/content/waveforge')
!git config user.name 'Shahzaib Ur Rehman'
!git config user.email 'shahzaib@waveforge.io'

# Save results JSON
results = {
    'meta': {'date': datetime.datetime.now().isoformat(), 'gpu': GPU_NAME,
             'platform': 'Google Colab T4', 'scenario': 'brain_clot_mimo',
             'grid': f'{NX}x{NY}', 'n_tx': N_TX, 'n_steps': N_STEPS},
    'waveforge_gpu': gpu_result,
    'meep_cpu':      meep_brain_result,
    'speedup_throughput': round(speedup, 2),
    'speedup_time':       round(time_speedup, 2),
}
with open('benchmarks/brain_clot_gpu_results.json', 'w') as f:
    json.dump(results, f, indent=2)

shutil.copy('/content/brain_clot_comparison.png', 'assets/brain_clot_comparison.png')

if GITHUB_TOKEN:
    import subprocess
    subprocess.run(['git','remote','set-url','origin',
                    f'https://{GITHUB_TOKEN}@github.com/shahzaibshazoo/waveforge.git'])
    !git add benchmarks/brain_clot_gpu_results.json assets/brain_clot_comparison.png
    !git commit -m 'Add real brain clot MIMO benchmark: GPU vs Meep'
    !git push origin main
    print('Results pushed to GitHub!')
else:
    print('Set GITHUB_TOKEN to auto-push, or download files manually')
    print('Files ready: benchmarks/brain_clot_gpu_results.json')
    print('            assets/brain_clot_comparison.png')


In [ ]:
# Summary
from IPython.display import Image, display
print(f'Brain Clot MIMO: WaveForge GPU {speedup:.1f}x faster throughput than Meep CPU')
print(f'Full 16-TX simulation: {gpu_result["t_full_mimo_s"]:.1f}s (GPU) vs {meep_brain_result["t_full_mimo_s"]:.1f}s (Meep CPU)')
display(Image('/content/brain_clot_comparison.png'))
